### Step:1 Importing all the libraries 


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

pd.set_option('display.max_columns', None)

### Step 2: Loading the datasets 


In [2]:
df = pd.read_csv("dataset/earthquake.csv")

### Step 3: Look at the dataset first

In [3]:
print("Shape of dataset (rows, columns):", df.shape)
print("\nFirst 5 rows of dataset:")
print(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values in each column:")
print(df.isnull().sum())

print("\nEvent type counts:")
print(df["type"].value_counts())

Shape of dataset (rows, columns): (17100, 22)

First 5 rows of dataset:
                       time  latitude  longitude    depth  mag magType    nst  \
0  2026-05-30T21:15:59.807Z   38.3564    73.8219  131.794  4.9      mb   80.0   
1  2026-05-29T12:22:54.128Z   23.4621    93.7220   57.374  4.3      mb   32.0   
2  2026-05-28T02:37:50.460Z   33.1393    96.1517   10.000  4.7      mb  111.0   
3  2026-05-26T16:30:57.714Z   23.1849    94.5425  102.712  4.3      mb   21.0   
4  2026-05-26T14:07:51.354Z   23.7982    94.8304  112.776  4.5      mb   36.0   

     gap   dmin   rms net          id                   updated  \
0   78.0  1.052  0.94  us  us7000spkb  2026-05-30T21:35:21.040Z   
1  126.0  2.503  0.68  us  us7000sp9a  2026-05-29T12:58:49.040Z   
2   47.0  5.492  0.73  us  us7000snzw  2026-05-28T05:26:32.040Z   
3  158.0  2.000  0.43  us  us7000sntk  2026-05-27T14:59:41.040Z   
4   68.0  2.581  0.40  us  us6000t08d  2026-05-26T17:07:05.040Z   

                                   pla

### STEP 4: DATA PREPROCESSING
#### latitude, longitude and depth (these describe WHERE the earthquake happened) we also keep magnitude to study the clusters later

### 4.1 Selecting clustering features. latitude, longitude and depth describe WHERE the earthquake happened, which is exactly what K-Means needs to find spatial zones. mag is kept alongside (not used to form the clusters) so we can study how magnitude behaves within each discovered zone afterwards.

In [4]:
print("Rows before filtering to earthquakes only:", df.shape[0])
df = df[df["type"] == "earthquake"].copy()
print("Rows after removing non-earthquake events (e.g. nuclear explosions):", df.shape[0])

columns_needed = ["latitude", "longitude", "depth", "mag", "place"]
df_cluster = df[columns_needed].copy()

Rows before filtering to earthquakes only: 17100
Rows after removing non-earthquake events (e.g. nuclear explosions): 17097


### Remove rows that have missing values (simple way to handle missing data)

In [5]:
print("\nRows before removing missing values:", df_cluster.shape[0])
df_cluster = df_cluster.dropna()
print("Rows after removing missing values:", df_cluster.shape[0])


Rows before removing missing values: 17097
Rows after removing missing values: 17097


### Remove duplicate rows if any

In [6]:
df_cluster = df_cluster.drop_duplicates()
print("Rows after removing duplicates:", df_cluster.shape[0])
df_cluster = df_cluster.reset_index(drop=True)

Rows after removing duplicates: 17097


### STEP 5: FEATURE ENGINEERING

## Step 5: Feature Engineering

### Two things happen here:

##### 1. **Scaling** — latitude, longitude and depth are on very different numeric ranges (degrees vs. kilometres). K-Means uses Euclidean distance, so without scaling, depth (which can reach ~400) would dominate the distance calculation and drown out the lat/long signal. StandardScaler puts all three features on the same footing (mean 0,  standard deviation 1).

##### 2. **Engineered feature — distance from Kathmandu.** To connect the clustering results directly back to Nepal, we compute the great-circle (Haversine) distance of every earthquake from Kathmandu (27.7172° N, 85.3240° E). This is not fed into the K-Means model itself (it would just duplicate the lat/long signal), but it is used afterwards to identify which discovered cluster represents the seismic zone closest to Nepal.

In [7]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance (km) between two lat/long points."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

KATHMANDU_LAT, KATHMANDU_LON = 27.7172, 85.3240
df_cluster["dist_from_kathmandu_km"] = haversine_km(
    df_cluster["latitude"], df_cluster["longitude"], KATHMANDU_LAT, KATHMANDU_LON
)

features = df_cluster[["latitude", "longitude", "depth"]]

scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

print("\nFeatures before scaling (first 5 rows):")
print(features.head())

print("\nFeatures after scaling (first 5 rows):")
print(features_scaled[:5])


Features before scaling (first 5 rows):
   latitude  longitude    depth
0   38.3564    73.8219  131.794
1   23.4621    93.7220   57.374
2   33.1393    96.1517   10.000
3   23.1849    94.5425  102.712
4   23.7982    94.8304  112.776

Features after scaling (first 5 rows):
[[ 0.98965357 -0.50998069  1.12784574]
 [-1.59964625  1.33987507 -0.0583901 ]
 [ 0.08268674  1.56573296 -0.81351959]
 [-1.64783609  1.41614638  0.66428607]
 [-1.54121694  1.44290873  0.82470366]]


### STEP 6: FIND THE BEST NUMBER OF CLUSTERS (K)
###  We use the Elbow Method to choose a good value of K
### We try K = 2 to 10 and check the "inertia" (how tight the clusters are)


In [8]:
inertia_values = []
k_range = range(2, 11)

for k in k_range:
    kmeans_test = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_test.fit(features_scaled)
    inertia_values.append(kmeans_test.inertia_)

print("\nInertia values for K = 2 to 10:")
for k, inertia in zip(k_range, inertia_values):
    print(f"K = {k} -> Inertia = {inertia:.2f}")


Inertia values for K = 2 to 10:
K = 2 -> Inertia = 27038.19
K = 3 -> Inertia = 15927.18
K = 4 -> Inertia = 11971.73
K = 5 -> Inertia = 9046.35
K = 6 -> Inertia = 7120.44
K = 7 -> Inertia = 6113.85
K = 8 -> Inertia = 5318.35
K = 9 -> Inertia = 4722.49
K = 10 -> Inertia = 4194.08


### We also check the Silhouette Score for a few K values
### Silhouette score tells us how well separated the clusters are (closer to 1 is better)

In [9]:
print("\nSilhouette scores for K = 2 to 8:")
silhouette_scores = []
for k in range(2, 9):
    kmeans_test = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_test = kmeans_test.fit_predict(features_scaled)
    score = silhouette_score(features_scaled, labels_test)
    silhouette_scores.append(score)
    print(f"K = {k} -> Silhouette Score = {score:.4f}")


Silhouette scores for K = 2 to 8:
K = 2 -> Silhouette Score = 0.4492
K = 3 -> Silhouette Score = 0.4710
K = 4 -> Silhouette Score = 0.4422
K = 5 -> Silhouette Score = 0.4620
K = 6 -> Silhouette Score = 0.4751
K = 7 -> Silhouette Score = 0.4700
K = 8 -> Silhouette Score = 0.4667


### Based on the elbow method and silhouette score, we choose our final K
### (After checking both results, K = 5 was selected for this project)

In [10]:
best_k = 5

### STEP 7: TRAIN THE FINAL K-MEANS MODEL

In [11]:
kmeans_model = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df_cluster["cluster"] = kmeans_model.fit_predict(features_scaled)

print(f"\nFinal model trained with K = {best_k}")
print("\nNumber of earthquakes in each cluster:")
print(df_cluster["cluster"].value_counts().sort_index())


Final model trained with K = 5

Number of earthquakes in each cluster:
cluster
0    3617
1    3603
2    5513
3    2927
4    1437
Name: count, dtype: int64


### STEP 8: EVALUATE THE MODEL

In [12]:
final_silhouette = silhouette_score(features_scaled, df_cluster["cluster"])
print(f"\nFinal Silhouette Score for K = {best_k}: {final_silhouette:.4f}")

# Show the center of each cluster (in original scale, not scaled values)
cluster_centers_scaled = kmeans_model.cluster_centers_
cluster_centers_original = scaler.inverse_transform(cluster_centers_scaled)

print("\nCluster centers (latitude, longitude, depth):")
centers_df = pd.DataFrame(cluster_centers_original, columns=["latitude", "longitude", "depth"])
print(centers_df)


Final Silhouette Score for K = 5: 0.4620

Cluster centers (latitude, longitude, depth):
    latitude  longitude       depth
0  36.749993  71.316819  165.425984
1  31.697820  89.343696   21.537783
2  36.949148  72.808426   34.298867
3  23.149873  94.964390   50.014554
4  27.740890  67.336654   22.410435


### STEP 9: ANALYZE CLUSTERS

Group each earthquake by depth (same rules as random forest):
- **Shallow** → 0–70 km
- **Medium** → 70–300 km
- **Deep** → >300 km

Then summarize what each cluster looks like.

In [15]:
def classify_depth(depth):
    if depth <= 70:
        return "Shallow"
    elif depth <= 300:
        return "Medium"
    return "Deep"

df_cluster["depth_class"] = df_cluster["depth"].apply(classify_depth)

cluster_summary = df_cluster.groupby("cluster").agg(
    earthquakes=("mag", "count"),
    avg_mag=("mag", "mean"),
    avg_depth_km=("depth", "mean"),
    max_mag=("mag", "max"),
    avg_dist_from_ktm_km=("dist_from_kathmandu_km", "mean"),
    main_place=("place", lambda x: x.value_counts().index[0]),
    main_depth_type=("depth_class", lambda x: x.value_counts().index[0]),
).round(2)

print("=== CLUSTER SUMMARY ===")
print(cluster_summary)

print("\n=== DEPTH CLASS COUNT PER CLUSTER ===")
depth_breakdown = pd.crosstab(df_cluster["cluster"], df_cluster["depth_class"])
print(depth_breakdown[["Shallow", "Medium", "Deep"]])

nearest_cluster = cluster_summary["avg_dist_from_ktm_km"].idxmin()
print(f"\nCluster nearest to Nepal: Cluster {nearest_cluster}")

=== CLUSTER SUMMARY ===
         earthquakes  avg_mag  avg_depth_km  max_mag  avg_dist_from_ktm_km  \
cluster                                                                      
0               3617     4.52        165.43      7.5               1657.87   
1               3603     4.65         21.55      7.8                753.72   
2               5513     4.59         34.30      7.6               1582.63   
3               2927     4.61         50.00      7.7               1132.76   
4               1437     4.63         22.38      7.7               1817.11   

                             main_place main_depth_type  
cluster                                                  
0        Hindu Kush region, Afghanistan          Medium  
1                        western Xizang         Shallow  
2              southern Xinjiang, China         Shallow  
3                               Myanmar         Shallow  
4             Owen Fracture Zone region         Shallow  

=== DEPTH CLASS COUNT 

### STEP 10: SAVE THE RESULTS

In [14]:
df_cluster.to_csv("dataset/results/earthquake_with_clusters.csv", index=False)
print("\nClustered data saved to earthquake_with_clusters.csv")

print("\n=== K-MEANS CLUSTERING COMPLETE ===")


Clustered data saved to earthquake_with_clusters.csv

=== K-MEANS CLUSTERING COMPLETE ===
